# Logistic Regression and GLMs

**Abstract**: Naive Bayes assumptions allow us to specify class
conditional densities through assuming that the data are conditionally
independent given parameters. A logistic regression is an approach to
classification which extends the linear basis function models we’ve
already explored. Rather than modeling the output of the function
directly the assumption is that we model the *log-odds* with the basis
functions.

$$
$$

<!-- CIP-0012 Track A: no-op outside HTML (overridden in talk-macros-html.gpp). -->
<!-- Class-test timer: no-op outside slides HTML (overridden in talk-macros-slides-html.gpp). -->
<!-- CIP-0016: mdpp expands \diagramcredit before gpp; leftover calls are empty. -->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!---->
<!-- CIP-0012 Track A: content script load. Once-guard with alphanumeric
     names stays in the surrounding ifndef wrapper (GPP ifndef limitation). -->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!-- The last names to be defined. Should be defined entirely in terms of macros from above-->
<!--

-->

## Setup

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_notebooks/includes/notebook-setup.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_notebooks/includes/notebook-setup.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import matplotlib.pyplot as plt
import shutil

if shutil.which('latex') is None:
    plt.rcParams['text.usetex'] = False
else:
    plt.rcParams['text.usetex'] = True
    plt.rcParams['text.latex.preamble'] = (
        r'\usepackage{amsmath}\usepackage{amssymb}\usepackage{mathrsfs}'
    )

plt.rcParams.update({'font.size': 22})

<!--setupplotcode{import seaborn as sns
sns.set_style('darkgrid')
sns.set_context('paper')
sns.set_palette('colorblind')}-->

## notutils

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_software/includes/notutils-software.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_software/includes/notutils-software.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

This small package is a helper package for various notebook utilities
used below.

The software can be installed using

In [ ]:
import importlib.util

In [ ]:
def install_command(imp):
    packages = {
        'daft': 'daft-pgm',
        'mlai': 'git+https://github.com/lawrennd/mlai.git',
        'pods': 'pods',
        'PyDeepGP': 'git+https://github.com/SheffieldML/PyDeepGP.git',
        'notutils': 'git+https://github.com/lawrennd/notutils.git',
        'pods' : 'git+https://github.com/lawrennd/ods.git',
        'lamd': 'lamd',
        'GPy': 'gpy',
    'emukit': 'emukit',
    }

    if importlib.util.find_spec(imp) is None:
        if imp in packages:
            return f'pip install {packages[imp]}'
        else:
            return f'pip install {imp}'
    else:
        return f'echo "Package {imp} is already installed."'

In [ ]:
cmd = install_command('notutils')

In [ ]:
%system {cmd}

from the command prompt where you can access your python installation.

The code is also available on GitHub:
<https://github.com/lawrennd/notutils>

Once `notutils` is installed, it can be imported in the usual manner.

In [ ]:
import notutils

## pods

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_software/includes/pods-software.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_software/includes/pods-software.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In Sheffield we created a suite of software tools for ‘Open Data
Science.’ Open data science is an approach to sharing code, models and
data that should make it easier for companies, health professionals and
scientists to gain access to data science techniques.

You can also check this blog post on [Open Data
Science](http://inverseprobability.com/2014/07/01/open-data-science).

The software can be installed using

In [ ]:
cmd = install_command('pods')

In [ ]:
%system {cmd}

from the command prompt where you can access your python installation.

The code is also available on GitHub: <https://github.com/lawrennd/ods>

Once `pods` is installed, it can be imported in the usual manner.

In [ ]:
import pods

## mlai

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_software/includes/mlai-software.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_software/includes/mlai-software.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

The `mlai` software is a suite of helper functions for teaching and
demonstrating machine learning algorithms. It was first used in the
Machine Learning and Adaptive Intelligence course in Sheffield in 2013.

The software can be installed using

In [ ]:
cmd = install_command('mlai')

In [ ]:
%system {cmd}

from the command prompt where you can access your python installation.

The code is also available on GitHub: <https://github.com/lawrennd/mlai>

Once `mlai` is installed, it can be imported in the usual manner.

In [ ]:
import mlai
from mlai import plot

## Review

The allowed us to specify a class conditional density,
$p(\mathbf{ x}_i|y_i, \boldsymbol{ \theta})$, through assuming that the
features were conditionally independent given the label. Combined with
our assumption that the data points are conditionally independent given
the parameters, $\boldsymbol{ \theta}$, this allowed us to specify a
joint density over the entire data set, $p(\mathbf{ y}, \mathbf{X})$. We
argued that modeling the joint density is a powerful approach because we
can answer any particular question we have about the data through the
sum rule and the product rule of probability. We can condition on the
training data and query the value of an unseen test point. If we have
missing data, then we can integrate over the missing point (marginalise)
and obtain our best prediction despite the absence of some of the
features for a point. However, it comes at the cost of a particular
modeling assumption. Namely, to make modeling practical we assumed that
the features were conditionally independent given the feature label. In
other words, for any given point, if we know its class, then its
features will be independent. This is a very strong assumption. For
example, if we were classifying the sex of an individual given their
height and weight, naive Bayes would assume that if we knew their sex,
then the height and weight would be independent. This is clearly wrong,
the dependence between height and weight is not dictated only by the sex
of an individual, there is a natural correlation between them.

Modeling the entire joint density allows us to deal with different
questions, that we may not have envisaged at the model *design time*. It
contrasts with the approach we took for regression where we specifically
chose to model the conditional density for the target values,
$\mathbf{ y}$, given the input locations, $\mathbf{X}$. That density,
$p(\mathbf{ y}|\mathbf{X})$, effectively assumes that the question we’ll
be asked at *run time* is known. In particular, we expect to be asked
about the value of the function, $y^*$, given a particular input
location, $\mathbf{ x}^*$. We don’t expect to be asked about the value
of an input given a particular observation. That would require placing
an additional prior over the input location for each point,
$p(\mathbf{ x}_i)$. Of course, it’s possible to conceive of a model like
this, and indeed that is how we proceeded for . However, if we know we
will always have all the inputs at run time, it may make sense to
*directly* model the conditional density, $p(\mathbf{ y}|\mathbf{X})$.

## Logistic Regression

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-intro.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-intro.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

A logistic regression is an approach to classification which extends the
linear regression models we’ve already explored. Rather than modeling
the output of the function directly the assumption is that we model the
*log-odds* with the basis functions.

The [odds](http://en.wikipedia.org/wiki/Odds) are defined as the ratio
of the probability of a positive outcome, to the probability of a
negative outcome. Just as we saw in our jumper (sweater) example where
$$ 
\log \frac{p(\text{bought})}{p(\text{not bought})} = w_0 + w_1 \text{age} + w_2 \text{latitude} 
$$ If the probability of a positive outcome is denoted by $\pi$, then
the odds are computed as $\frac{\pi}{1-\pi}$.

Odds are widely used by
[bookmakers](http://en.wikipedia.org/wiki/Bookmaker) in gambling,
although a bookmakers odds won’t normalise: i.e. if you look at the
equivalent probabilities, and sum over the probability of all outcomes
the bookmakers are considering, then you won’t get one. This is how the
bookmaker makes a profit. Because a probability is always between zero
and one, the odds are always between $0$ and $\infty$. If the positive
outcome is unlikely the odds are close to zero, if it is very likely
then the odds become close to infinite. Taking the logarithm of the odds
maps the odds from the positive half space to being across the entire
real line. Odds that were between 0 and 1 (where the negative outcome
was more likely) are mapped to the range between $-\infty$ and $0$. Odds
that are greater than 1 are mapped to the range between $0$ and
$\infty$. Considering the log odds therefore takes a number between 0
and 1 (the probability of positive outcome) and maps it to the entire
real line. The function that does this is known as the [logit
function](http://en.wikipedia.org/wiki/Logit),
$g(p_i) = \log\frac{p_i}{1-p_i}$. This function is known as a *link
function*.

For a standard regression we take, $$
f(\mathbf{ x}) = \mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x}),
$$ if we want to perform classification we perform a logistic
regression. $$
\log \frac{\pi}{(1-\pi)} = \mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x})
$$ where the odds ratio between the positive class and the negative
class is given by $$
\frac{\pi}{(1-\pi)}
$$ The odds can never be negative, but can take any value from 0 to
$\infty$. We have defined the link function as taking the form
$g^{-1}(\cdot)$ implying that the inverse link function is given by
$g(\cdot)$. Since we have defined, $$
g(\pi) =
\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x})
$$ we can write $\pi$ in terms of the *inverse link* function,
$h(\cdot)$ as $$
\pi = h(\mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x})).
$$

In [ ]:
import mlai.plot as plot

In [ ]:
plot.logistic('./ml')

We’ll define our prediction, objective and gradient functions below. But
before we start, we need to define a basis function for our model. Let’s
start with the linear basis.

In [ ]:
import numpy as np

In [ ]:
from mlai import linear
import inspect
file_path = inspect.getfile(linear)

In [ ]:
%load -s linear {file_path}

## Soft Arg Max (Softmax) Function

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/soft-arg-max-function.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/soft-arg-max-function.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Given class scores
$f_k(\mathbf{ x})=\mathbf{ w}_k^\top\boldsymbol{ \phi}(\mathbf{ x})$,
the softmax maps scores to probabilities:
$\pi_k = \frac{\exp(f_k)}{\sum_{j=1}^K \exp(f_j)}$.

## Prediction Function

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-prediction-function.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-prediction-function.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Now we have the basis function let’s define the prediction function.

In [ ]:
import numpy as np

In [ ]:
def predict(w, x, basis=mlai.linear, **kwargs):
    "Generates the prediction function and the basis matrix."
    Phi = basis(x, **kwargs)
    f = np.dot(Phi, w)
    return 1./(1+np.exp(-f)), Phi

This inverse of the link function is known as the
[logistic](http://en.wikipedia.org/wiki/Logistic_function) (thus the
name logistic regression) or sometimes it is called the sigmoid
function. For a particular value of the input to the link function,
$f_i = \mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x}_i)$ we can plot
the value of the inverse link function as below.

By replacing the inverse link with the sigmoid we can write $\pi$ as a
function of the input and the parameter vector as, $$
\pi(\mathbf{ x},\mathbf{ w}) = \frac{1}{1+\exp\left(-\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x})\right)}.
$$ The process for logistic regression is as follows. Compute the output
of a standard linear basis function composition
($\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x})$, as we did for
linear regression) and then apply the inverse link function,
$g(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x}))$. In logistic
regression this involves *squashing* it with the logistic (or sigmoid)
function. Use this value, which now has an interpretation as a
*probability* in a Bernoulli distribution to form the likelihood. Then
we can assume conditional independence of each data point given the
parameters and develop a likelihod for the entire data set.

The Bernoulli likelihood is of the form, $$
P(y_i|\mathbf{ w}, \mathbf{ x}) =
\pi_i^{y_i} (1-\pi_i)^{1-y_i}
$$ which we can think of as clever trick for mathematically switching
between two probabilities if we were to write it as code it would be
better described as

``` python
def bernoulli(x, y, pi):
    if y == 1:
        return pi(x)
    else:
        return 1-pi(x)
```

but writing it mathematically makes it easier to write our objective
function within a single mathematical equation.

## Maximum Likelihood

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-maximum-likelihood.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-maximum-likelihood.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

To obtain the parameters of the model, we need to maximise the
likelihood, or minimise the objective function, normally taken to be the
negative log likelihood. With a data conditional independence assumption
the likelihood has the form, $$
P(\mathbf{ y}|\mathbf{ w},
\mathbf{X}) = \prod_{i=1}^nP(y_i|\mathbf{ w}, \mathbf{ x}_i). 
$$ which can be written as a log likelihood in the form $$
\log P(\mathbf{ y}|\mathbf{ w},
\mathbf{X}) = \sum_{i=1}^n\log P(y_i|\mathbf{ w}, \mathbf{ x}_i) = \sum_{i=1}^n
y_i \log \pi_i + \sum_{i=1}^n(1-y_i)\log (1-\pi_i)
$$ and if we take the probability of positive outcome for the $i$th data
point to be given by $$
\pi_i = h\left(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x}_i)\right),
$$ where $h(\cdot)$ is the *inverse* link function, then this leads to
an objective function of the form, $$
E(\mathbf{ w}) = -  \sum_{i=1}^ny_i \log
h\left(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x}_i)\right) -
\sum_{i=1}^n(1-y_i)\log \left(1-h\left(\mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x}_i)\right)\right).
$$

## Negative Log Likelihood

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-optimisation.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-optimisation.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import numpy as np

In [ ]:
def objective(h, y):
    "Computes the objective function."
    labs = np.asarray(y, dtype=float).flatten()
    posind = np.where(labs==1)
    negind = np.where(labs==0)
    return -np.log(h[posind, :]).sum() - np.log(1-h[negind, :]).sum()

As normal, we would like to minimise this objective. This can be done by
differentiating with respect to the parameters of our prediction
function, $\pi(\mathbf{ x};\mathbf{ w})$, for optimisation. The gradient
of the likelihood with respect to $\pi(\mathbf{ x};\mathbf{ w})$ is of
the form, $$
\frac{\text{d}E(\mathbf{ w})}{\text{d}\mathbf{ w}} = -\sum_{i=1}^n
\frac{y_i}{h\left(\mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x})\right)}\frac{\text{d}h(f_i)}{\text{d}f_i}
\boldsymbol{ \phi}(\mathbf{ x}_i) +  \sum_{i=1}^n
\frac{1-y_i}{1-h\left(\mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x})\right)}\frac{\text{d}h(f_i)}{\text{d}f_i}
\boldsymbol{ \phi}(\mathbf{ x}_i)
$$ where we used the chain rule to develop the derivative in terms of
$\frac{\text{d}h(f_i)}{\text{d}f_i}$, which is the gradient of the
inverse link function (in our case the gradient of the sigmoid
function).

So the objective function now depends on the gradient of the inverse
link function, as well as the likelihood depends on the gradient of the
inverse link function, as well as the gradient of the log likelihood,
and naturally the gradient of the argument of the inverse link function
with respect to the parameters, which is simply
$\boldsymbol{ \phi}(\mathbf{ x}_i)$.

The only missing term is the gradient of the inverse link function. For
the sigmoid squashing function we have,

so the full gradient can be written down as $$
\frac{\text{d}E(\mathbf{ w})}{\text{d}\mathbf{ w}} = -\sum_{i=1}^n
y_i\left(1-h\left(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x})\right)\right)
\boldsymbol{ \phi}(\mathbf{ x}_i) +  \sum_{i=1}^n
(1-y_i)\left(h\left(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x})\right)\right)
\boldsymbol{ \phi}(\mathbf{ x}_i).
$$

In [ ]:
import numpy as np

In [ ]:
def gradient(h, Phi, y):
    "Generates the gradient of the parameter vector."
    labs = np.asarray(y, dtype=float).flatten()
    posind = np.where(labs==1)
    dw = -(Phi[posind]*(1-h[posind])).sum(0)
    negind = np.where(labs==0 )
    dw += (Phi[negind]*h[negind]).sum(0)
    return dw[:, None]

## Optimisation of the Function

Reorganising the gradient to find a stationary point of the function
with respect to the parameters $\mathbf{ w}$ turns out to be impossible.
Optimisation has to proceed by *numerical methods*. Options include the
multidimensional variant of [Newton’s
method](http://en.wikipedia.org/wiki/Newton%27s_method) or [gradient
based optimisation
methods](http://en.wikipedia.org/wiki/Gradient_method) like we used for
optimising matrix factorisation for the movie recommender system. We
recall from matrix factorisation that, for large data, *stochastic
gradient descent* or the Robbins Munro (Robbins and Monro, 1951)
optimisation procedure worked best for function minimisation.

Multiplying everything out leads to $$
\frac{\text{d}E(\mathbf{ w})}{\text{d}\mathbf{ w}} = -\sum_{i=1}^n\left(y_i - h\left(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x}_i)\right)\right)\boldsymbol{ \phi}(\mathbf{ x}_i).
$$

## Olivetti Glasses Data

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/olivetti-glasses-logistic.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/olivetti-glasses-logistic.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Let’s classify images with logistic regression. We’ll look at a data set
of individuals with glasses. We can load in the data from `pods` as

Correspond to whether the subject of the image is wearing glasses. Set
up the ipython environment and download the data:

In [ ]:
from scipy import io

First let’s retrieve some data. We will use the ORL faces data set, our
objective will be to classify whether or not the subject in an image is
wearing glasess.

Here’s a simple way to visualise the data. Each pixel in the image will
become an input to the GP.

In [ ]:
import pods

In [ ]:
data = pods.datasets.olivetti_glasses()
X = data['X']
y = data['Y']
Xtest = data['Xtest']
ytest = data['Ytest']
print(data['info'], data['details'], data['citation'])

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
ax.imshow(X[0].reshape(64, 64, order='F'),
          interpolation='nearest',
          cmap=plt.cm.gray)

mlai.write_figure("olivetti-glasses-image.png", 
                  directory="./datasets")

<img class="" src="https://mlatcl.github.io/mlai/slides/diagrams/datasets/olivetti-glasses-image.png" style="width:60%; height:auto; display:block;">

Figure: <i>Image from the Oivetti glasses data sets.</i>

## Batch Gradient Descent

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-gradient-descent.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-gradient-descent.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We will need to define some initial random values for our vector and
then minimize the objective by descending the gradient.

In [ ]:
# Separate train and test
indices = np.random.permutation(X.shape[0])
num_train = int(np.ceil(X.shape[0]/2))
train_indices = indices[:num_train]
test_indices = indices[num_train:]
if hasattr(X, "iloc"):
    X_train = X.iloc[train_indices]
    X_test = X.iloc[test_indices]
else:
    X_train = X[train_indices]
    X_test = X[test_indices]
if hasattr(y, "iloc"):
    y_train = y.iloc[train_indices]==True
    y_test = y.iloc[test_indices]==True
else:
    y_train = np.asarray(y).flatten()[train_indices]==True
    y_test = np.asarray(y).flatten()[test_indices]==True

In [ ]:
import numpy as np

In [ ]:
# gradient descent algorithm
w_vals = np.random.normal(size=(X.shape[1]+1, 1), scale = 0.001)
# Convert weights to a pandas DataFrame with column names
if type(X_train) is pd.DataFrame:
    w = pd.DataFrame(data=w_vals, 
                     index=['Eins'] + list(X_train.columns), 
                     columns=['weight'])
else:
    w = w_vals

eta = 1e-9
iters = 10000
for i in range(iters):
    g, Phi = predict(w, X_train, mlai.linear)
    w -= eta*gradient(g, Phi, y_train) + 0.001*w
    if not i % 100:
        print("Iter", i, "Objective", objective(g, y_train))

## Going Further: Optimization

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-going-further.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-going-further.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Other optimization techniques for generalized linear models include
[Newton’s method](http://en.wikipedia.org/wiki/Newton%27s_method), it
requires you to compute the Hessian, or second derivative of the
objective function.

Methods that are based on gradients only include
[L-BFGS](http://en.wikipedia.org/wiki/Limited-memory_BFGS) and
[conjugate
gradients](http://en.wikipedia.org/wiki/Conjugate_gradient_method). Can
you find these in python? Are they suitable for very large data sets? }

## Bayesian Approaches

### Exercise 1

Can you place a prior density over the parameters $\mathbf{ w}$ and
marginalize them out like we did for linear regression? If not why not?

### Exercise 1 Answer

Write your answer to Exercise 1 here

In [ ]:
# Use this box for any code you need



## Further Reading

-   Section 5.2.2 up to pg 182 of Rogers and Girolami (2011)

## Thanks!

For more information on these subjects and more you might want to check
the following resources.

-   company: [Trent AI](https://trent.ai)
-   book: [The Atomic
    Human](https://www.penguin.co.uk/books/455130/the-atomic-human-by-lawrence-neil-d/9780241625248)
-   twitter: [@lawrennd](https://twitter.com/lawrennd)
-   podcast: [The Talking Machines](http://thetalkingmachines.com)
-   newspaper: [Guardian Profile
    Page](http://www.theguardian.com/profile/neil-lawrence)
-   blog:
    [http://inverseprobability.com](http://inverseprobability.com/blog.html)

## References

Robbins, H., Monro, S., 1951. A stochastic approximation method. Annals
of Mathematical Statistics 22, 400–407.

Rogers, S., Girolami, M., 2011. A first course in machine learning. CRC
Press.